# W03｜AI 研究環境建置：Google Colab、Gemini、Cursor Pro

**人工智慧於醫學影像組學的分析與應用（3010040）**　國防醫學院醫學科學研究所　林彥聖  
**授課日期：2026-09-17｜實作工作坊 3 小時**

---

今天下課時，你要帶走一個**別人也跑得起來，而且跑得出一樣數字**的研究環境。

本 notebook 對應講義的「出發前確認」與六個檢查點。每個技術檢查點都有程式與畫面判準；
對不上的地方，依課堂規則：**卡住三分鐘就舉手**。

| # | 檢查點 | 檢核方式 |
|---|---|---|
| 0 | 筆電、固定 Google 帳號、`w03_setup`、配對夥伴到位 | 手動 |
| 1 | GPU 啟用，並用指令驗證它真的在 | 自動＋畫面確認 |
| 2 | Drive 掛載成功，路徑用 `ROOT` 組合，沒有寫死 | 自動 |
| 3 | 與夥伴配對完成，兩台電腦都跑得動 | 手動 |
| 4 | 版本鎖定、種子設定、環境紀錄齊全 | 自動＋人工檢視 |
| 5 | Gemini 校準過、Cursor 開得了專案資料夾 | 手動 |
| 6 | 命名、六層資料夾與去識別化原則符合約定 | 自動＋口頭確認 |

> **開始之前**：把這本 notebook 另存到自己的 Google Drive，檔名設為 `w03_setup`。
> 不要留 `Untitled0`；命名是今天的第一條規矩。

> **資料安全紅線**：不得把可識別的病人資料貼進 Gemini、Cursor 或任何線上 AI 工具。

## 檢查點 0｜出發前確認

四項都到位才往下走。沒用過 Colab 是預設起點；運算在雲端進行，舊筆電或 Intel Mac 不影響今天的實作。

- [ ] 筆電已接上電源並連上穩定網路
- [ ] Google 帳號已登入，而且今天固定使用同一個帳號
- [ ] 已在 Colab 開啟並命名 `w03_setup`
- [ ] 身旁已有配對夥伴，稍後會輪流擔任駕駛與領航員

### 今日術語卡

| 術語 | 本課使用的意思 |
|---|---|
| GPU | 擅長大量平行計算的晶片；本課用指令確認是否真的可用 |
| runtime | Colab 提供的雲端電腦；斷線後套件、變數與 `/content` 會消失 |
| mount | 把 Google Drive 接到 Colab runtime |
| package / version | 別人寫好的工具與其版本；版本不同可能產生不同結果 |
| random seed | 固定隨機過程的設定 |
| reproducible | 別人依說明執行，也能得到相同結果 |
| tensor | 多維數字陣列；影像在程式中通常以張量表示 |
| de-identification | 移除所有可能辨識病人身分的資訊 |

### 工作坊規則

1. 卡住三分鐘就舉手。
2. 先讓流程跑起來，再改善外觀與寫法。
3. 配對時鍵盤要輪流；先完成者支援鄰座。

## 第一格：所有安裝指令都在這裡

為什麼堅持放同一格？因為**斷線是常態**。
斷線之後重跑這一格就全部恢復，不需要任何「我手動點了一下」的步驟 ——
手動的東西無法重放。

⚠️ 下面第一行的網址要換成課程的 GitHub 位址（我會在課堂上寫在白板）。

In [ ]:
# ── 第一格：環境安裝（斷線後只要重跑這一格） ──────────────────────

COURSE_REPO = "https://github.com/YOUR-GITHUB-ACCOUNT/ndmu-radiomics-ai.git"
REPO_DIR    = "ndmu-radiomics-ai"

import os, subprocess, sys

READY = False

if "YOUR-GITHUB-ACCOUNT" in COURSE_REPO:
    print("⚠️  上面第一行的 COURSE_REPO 還是預設值。")
    print("    請把 YOUR-GITHUB-ACCOUNT 換成課程 GitHub 帳號（白板上有），再重跑這一格。")
else:
    if not os.path.isdir(REPO_DIR):
        r = subprocess.run(["git", "clone", "-q", COURSE_REPO, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("⚠️  git clone 失敗。網址打錯，或這台機器連不到 GitHub。")
            print("    錯誤訊息：", (r.stderr or "").strip()[:300])

    if os.path.isdir(REPO_DIR):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "-r", os.path.join(REPO_DIR, "requirements.txt")],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("⚠️  套件安裝沒有全部成功。把下面整段貼到課程討論區，不要自己亂升級：")
            print((r.stderr or "").strip()[-800:])
        else:
            sys.path.insert(0, os.path.abspath(REPO_DIR))
            READY = True
            print("✓ 課程套件安裝完成，src 已經可以 import。")
            print("  若 pip 要求重新啟動執行階段，請重啟後【從這一格】重跑，不要從中間接著跑。")

if not READY:
    print()
    print("這一格沒有成功，下面的儲存格會 import 失敗，這是正常的 —— 先把這一格弄好。")
    print("卡住三分鐘就舉手。")

---
## 檢查點 1｜GPU 啟用，並用指令驗證

**先做**：執行階段／Runtime 選單 → 變更執行階段類型 → 硬體加速器選 GPU → 儲存。
右上角顯示「已連線」才算完成，通常十幾秒。

然後跑下面兩格。**不要相信介面，相信輸出。**

In [ ]:
# 第一層：這台虛擬機上有沒有實體 GPU？
# 驚嘆號代表這一行是給作業系統的命令，不是 Python。
!nvidia-smi

In [ ]:
# 第二層：你的框架看得到它嗎？回 True 才算通。
try:
    import torch
    print("torch", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
except ImportError:
    print("這台機器沒有裝 torch。")
    print("在 Colab 上 torch 是預裝的，出現這一行代表你不是在 Colab 跑 ——")
    print("這不影響今天的其他檢查點，繼續往下做。")

# 第三層（進階・課堂上我示範就好，你不用自己打）：
# 真的搬一個張量上去做一次運算，確認整條路是通的。
# x = torch.randn(1000, 1000, device="cuda")
# print((x @ x).sum().item())

### 你的螢幕應該長這樣

對照下面四項。任何一項對不上，手舉著不要放。

- ① 出現一張表格，最上面有驅動版本與 CUDA 版本
- ② 表格中間看得到顯示卡型號（型號因人而異，**有型號就對**）
- ③ 看得到記憶體用量，形如 `0MiB / 15360MiB`
- ④ 沒有出現 `command not found` 或空白輸出

**沒過的兩種狀況**（詳見 `docs/05_安裝疑難排解.md`）：

- **狀況 A**　`nvidia-smi` 有輸出但 `torch.cuda.is_available()` 回 `False` →
  不要升級 Colab 預裝的深度學習套件，重開執行階段再試。
- **狀況 B**　分不到 GPU → 免費版動態配額，忙碌時段會分不到。
  這是免費資源的本質，不是你操作錯。**今天所有練習 CPU 都跑得動。**

---
## 檢查點 2｜掛載 Drive，建立 ROOT

掛載只有兩行。難的是後面那個變數 —— 它決定你的 notebook 能不能給別人跑。

授權時請選**檢查點 0 決定的那一個帳號**。選錯之後權限會很麻煩。

In [ ]:
from src.paths import get_root, in_colab

ROOT = get_root(project="radiomics_course")   # Colab 會自動掛載 Drive
print("ROOT =", ROOT)

# 權限測試：實際寫一個小檔進去、再讀出來。不要用猜的。
probe = ROOT / "_write_test.txt"
probe.write_text("ok", encoding="utf-8")
print("讀回來：", probe.read_text(encoding="utf-8"))
probe.unlink()
print("權限測試通過")

### 你的螢幕應該長這樣

- ① `ROOT` 指到你自己 Drive 裡的資料夾
- ② notebook 最上面有一行 `ROOT = ...`，**底下沒有任何完整絕對路徑**
- ③ 我寫了一個小檔案進去、又讀出來了
- ④ 我知道 `/content` 底下的東西**斷線會消失**

> **斷線是常態。** 別問「怎麼避免斷線」，問「斷了之後我損失多少」。
> 中間結果一律存 Drive，長流程切成可獨立續跑的段落，中間用檔案接。

---
## 檢查點 3｜配對：兩人一機，把環境整條走一次　`手動`

駕駛負責打字，領航員負責看與說。**領航員不准碰鍵盤，駕駛不准自己想。**
中途我會喊換手。

兩個人各自在自己的螢幕上示範一次給對方看，互相確認完才休息：

- ① 兩台電腦都跑得出 `nvidia-smi` 輸出
- ② 兩台電腦都掛上了各自帳號的 Drive
- ③ 兩份 notebook 都有 `ROOT` 變數，且沒有寫死路徑
- ④ 我可以**不看筆記**把這條流程從頭講一遍給夥伴聽

---
## 檢查點 4｜版本鎖定、種子設定、環境紀錄

到現在為止，你的環境「跑得動」。接下來要讓它「**跑得出一樣的數字**」。
這兩件事的距離比你以為的遠。

**今天的要求只有一個**：第一格有 `set_seed(42)`，而且已經呼叫過。
為什麼要設三個位置的種子，現在不懂沒關係 —— 先照做，W10 比較模型時你會懂。

In [ ]:
from src.repro import set_seed, env_report, RANDOM_STATE
from src.paths import stamped

set_seed(42)

# 環境紀錄：跟結果存在一起。
# 半年後審稿人問「訓練用什麼硬體」，你要答得出來。
report = env_report(save_to=ROOT / "results" / stamped("env", ext=".json"))

### 往上捲到第一格，對照這四項

- ① 每一行 `pip install` 都帶 `==版本號`（本課程用 `requirements.txt` 一次搞定）
- ② 一個 `set_seed()`，而且開頭已經呼叫過
- ③ 環境紀錄：Python 版本、關鍵套件版本、GPU 型號、執行時間
- ④ 一行 `ROOT = ...`，底下沒有任何寫死的絕對路徑

> **最常漏的一項**：`set_seed()` 管不到 sklearn 的函式層級隨機性。
> 切分、交叉驗證、隨機森林、取樣 —— 看到 `random_state=` 就填 `RANDOM_STATE`：
> ```python
> train_test_split(X, y, random_state=RANDOM_STATE)
> RandomForestClassifier(random_state=RANDOM_STATE)
> ```

### 隨堂檢核｜先作答，再展開解析

**Q1. Colab 斷線後，下列哪一項一定不會留在 runtime？**  
A. `pip` 裝好的套件　B. Drive 裡的檔案　C. notebook 的程式碼儲存格

<details>
<summary>查看 Q1 答案與解析</summary>

**答案：A。** 斷線後虛擬機可能被回收，安裝的套件、變數與 `/content` 會消失；Drive 是外部儲存，程式碼則保存在 notebook 檔案中。因此安裝指令必須集中保留，讓環境可以重建。
</details>

**Q2. 下列哪一項最可能讓別人重現不出你的數字？**  
A. 沒寫註解　B. 變數名稱太短　C. 沒鎖套件版本

<details>
<summary>查看 Q2 答案與解析</summary>

**答案：C。** 套件升級可能改變特徵定義、預設參數或計算行為；註解與變數名稱主要影響可讀性，不會直接改變數值結果。
</details>

---
## 檢查點 5｜Gemini 與 Cursor　`手動`

**Gemini 今天只做三件事**（深入用法是 W05 的整堂內容）：

1. **確認登入** —— 用檢查點 0 那個帳號。學校帳號可能有機構限制。
2. **丟一個真的研究問題**，不要打「你好」。
   > 範例：「我有 120 例 CT 影像，想預測術後復發。這個樣本數夠嗎？驗證流程怎麼設計？」
3. **校準** —— 問一個你已經知道答案的專業細節，對照它答得對不對。
   目的是建立直覺：它在哪些主題可靠，在哪些主題會一本正經地胡說。

**Cursor**：下載安裝後，**開啟資料夾，不是開啟檔案** —— 以專案資料夾為單位，AI 才看得到全貌。
裝不了就改走純 Colab 路線，這也算通過。

> 🚫 **紅線：不得把可識別的病人資料貼進任何線上 AI 工具。**

---
## 檢查點 6｜資料夾結構與命名約定

約定的價值不在「哪一種比較好」，在「**大家都一樣**」。

In [ ]:
from src.paths import ensure_project, show_tree

ensure_project(ROOT)
show_tree(ROOT)

# 結果檔名照約定產生，不要每次手打、也不要每次覆寫同一個 result.csv
print()
print("結果檔名範例：", stamped("features_bin25"))

### 命名約定

| 規則 | 做法 |
|---|---|
| 檔名字元 | 只用英數字與底線，不用中文、空白、括號 |
| 日期 | `YYYYMMDD` 放最前面，排序就是時間順序 |
| notebook | `編號_動作_對象`，如 `01_preprocess_ct.ipynb` |
| 結果檔 | 帶日期與參數，不覆寫舊檔 |
| 病人 | 一律用代號 `SUB001`，**絕不使用病歷號或姓名** |

### 去識別化：技術上要做的三件事

1. **清除 DICOM 標頭** —— 姓名、病歷號、生日、檢查號、機構欄位。
   用程式批次處理，不要手動改（幾百個欄位，手動一定會漏）。
2. **檢查像素本身** —— 部分超音波／內視鏡把病人資訊燒進畫面。
   標頭清乾淨了，畫面左上角還有名字，這是最常被忽略的漏洞。
3. **對照表分開存放** —— 代號與真實身分的對照，不與影像同一個空間。

> **額外風險**：頭部 3D 影像可能重建出臉部，公開前還需執行去臉（defacing）。

### 檢查點 6 最後確認

- [ ] Drive 與本機都有相同的六層專案結構
- [ ] 檔名沒有中文或空白，病人只用 `SUB` 代號
- [ ] 能說出去識別化的三個技術步驟
- [ ] notebook 不含病人資料輸出，也沒有可識別資訊
- [ ] 把 notebook 交給同學，對方不提問即可執行

細節與可直接複製的程式碼在 `docs/03_去識別化檢核.md`。

---
## 下課前：跑一次自動檢核

In [ ]:
from src.checks import run_all

run_all(ROOT)

---
## 本週作業

**必做（三項）**

1. 六個檢查點全部完成，把這本 notebook 的連結繳交至課程平台
2. `docs/README.md` —— 課程 GitHub 有三行樣板（`docs/00_README樣板.md`），
   **填空即可，不要寫成正式文件**
3. 建立 GitHub 帳號（W06 實作，今天只要有帳號）

**建議・不計分**

- 與一位同學互測 —— 對方不問問題就跑得起來

**思考**

- 把 W02 想的臨床問題寫成一句話就好，不必寫成檢索式

**提醒**

- W10（11/05）期中計畫書還有 7 週；**沒有資料的人，W05 前寄三行給我**
  （手上有什麼、有多少、卡在哪）

---
下週　第 4 週｜醫學影像基礎：CT、MRI、PET、超音波、X-ray　
請帶上今天建好的環境，下週起每一週都會用到它。